# Ćwiczenia 2: Model ML, FastAPI i serwowanie predykcji

*Studia zaoczne | 1,5h*

## Cel

- Wytrenowanie modelu detekcji fraudów,
- Serwowanie modelu przez FastAPI,
- Podłączenie do Kafki — scoring w czasie rzeczywistym.

---

## Część 1: Trening modelu (25 min)

### Zadanie 1.1 — Przygotuj dane i wytrenuj model

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, 2000).clip(5, 5000),
    'hour': np.random.normal(14, 4, 2000).clip(0, 23).astype(int),
    'is_electronics': np.random.binomial(1, 0.3, 2000),
    'tx_per_day': np.random.poisson(3, 2000),
    'fraud': 0
})

# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, 100),
    'hour': np.random.choice([0,1,2,3,4,5,22,23], 100),
    'is_electronics': np.random.binomial(1, 0.7, 100),
    'tx_per_day': np.random.poisson(8, 100),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


### Zadanie 1.2 — Podziel dane, wytrenuj, oceń

In [2]:
features = ['amount', 'hour', 'is_electronics', 'tx_per_day']
X = df[features]
y = df['fraud']

# TWÓJ KOD
# 1. train_test_split (80/20, stratify=y)
# 2. RandomForestClassifier(100)
# 3. classification_report
# 4. pickle.dump do 'fraud_model.pkl'

# 1. Podział danych na zbiór treningowy i testowy (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 2. Inicjalizacja i trenowanie modelu RandomForest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 3. Predykcja i ocena modelu
y_pred = model.predict(X_test)
print("Raport klasyfikacji:")
print(classification_report(y_test, y_pred))

# 4. Zapisanie modelu do pliku 'fraud_model.pkl'
with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model został zapisany pomyślnie.")

Raport klasyfikacji:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420

Model został zapisany pomyślnie.


---

## Część 2: FastAPI (25 min)

### Zadanie 2.1 — Utwórz API serwujące model

In [3]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

# TWÓJ KOD
# Endpoint POST /score
# Przyjmij Transaction, zwróć: {"is_fraud": bool, "fraud_probability": float}

@app.post("/score")
def predict_fraud(transaction: Transaction):
    # 1. Konwersja danych z obiektu Pydantic na format akceptowany przez model (2D array)
    data = [[
        transaction.amount,
        transaction.hour,
        transaction.is_electronics,
        transaction.tx_per_day
    ]]
    
    # 2. Predykcja klasy (0 lub 1)
    prediction = model.predict(data)[0]
    
    # 3. Predykcja prawdopodobieństwa (wyciągamy wartość dla klasy 1 - fraud)
    probability = model.predict_proba(data)[0][1]
    
    # 4. Zwrócenie wyniku zgodnego z wymaganiami zadania
    return {
        "is_fraud": bool(prediction),
        "fraud_probability": float(probability)
    }

@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "model_loaded": model is not None,
        "timestamp": str(datetime.now())
    }

Writing fraud_api.py


Uruchom w terminalu:
```bash
uvicorn fraud_api:app --host 0.0.0.0 --port 8001
```

### Zadanie 2.2 — Przetestuj API

In [4]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "hour": 14, "is_electronics": 0, "tx_per_day": 3})
print("Normalna:", r.json())

# TWÓJ KOD — przetestuj podejrzaną transakcję (amount=5500, hour=3, is_electronics=1, tx_per_day=12)

# Test podejrzana
r_fraud = requests.post("http://localhost:8001/score",
    json={
        "amount": 5500, 
        "hour": 3, 
        "is_electronics": 1, 
        "tx_per_day": 12
    })
print("Podejrzana:", r_fraud.json())

ConnectionError: HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /score (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8001): Failed to establish a new connection: [Errno 61] Connection refused"))

---

## Część 3: Kafka + ML (25 min)

### Zadanie 3.1 — Konsument z scoringiem ML

Napisz konsumenta, który czyta z `transactions`, odpytuje API i wysyła alerty.

In [5]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

# TWÓJ KOD
# Dla każdej transakcji:
# 1. Wyciągnij cechy (amount, hour z timestamp, is_electronics, tx_per_day=5)
# 2. requests.post(API_URL, json=features)
# 3. Jeśli is_fraud: wyślij do tematu 'alerts', wypisz ALERT

for message in consumer:
    transaction = message.value
    
    # 1. Wyciągnij cechy (ekstrakcja godziny z timestampu)
    # Zakładamy, że timestamp jest w formacie ISO, np. "2026-04-18T08:30:00"
    dt = datetime.fromisoformat(transaction['timestamp'])
    
    features = {
        "amount": transaction['amount'],
        "hour": dt.hour,
        "is_electronics": transaction['is_electronics'],
        "tx_per_day": 5  # wartość stała zgodnie z poleceniem
    }
    
    # 2. Odpytanie API (Scoring)
    try:
        response = requests.post(API_URL, json=features)
        prediction = response.json()
        
        # 3. Jeśli model wykrył fraud, wyślij alert do tematu 'alerts'
        if prediction.get("is_fraud"):
            alert_payload = {
                "transaction_id": transaction.get("id", "N/A"),
                "amount": transaction['amount'],
                "probability": prediction.get("fraud_probability"),
                "timestamp": transaction['timestamp']
            }
            
            alert_producer.send('alerts', value=alert_payload)
            print(f"ALERT: Wykryto oszustwo! Prawdopodobieństwo: {prediction.get('fraud_probability'):.2f}")
            
    except Exception as e:
        print(f"Błąd podczas łączenia z API: {e}")

Writing ml_consumer.py


### Zadanie 3.2 — Uruchom pipeline

W 3 terminalach:
1. `python producer.py` (z Ćw. 1)
2. `uvicorn fraud_api:app --host 0.0.0.0 --port 8001`
3. `python ml_consumer.py`

Obserwuj alerty.

---

## Praca domowa

1. Porównaj wyniki scoringu regułowego (Ćw. 1) vs ML — który lepiej wykrywa?
2. Dodaj endpoint `GET /health` do API.
3. Wypchnij do Git.

**Następne zajęcia:** Apache Spark — przetwarzanie na dużą skalę.